# 第 5 章互动实验：角点与旋转重复性

本实验在合成标记上调用 OpenCV Harris 基线，观察阈值和旋转效果；不会给出 `5-1_Harris响应.py` 的结构张量实现。后面三个实验对应教程 5.2 节的实测，都要用到你自己写的 `harris_response`。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

base = np.zeros((180, 220), dtype=np.uint8)
for y in range(30, 151, 30):
    for x in range(35, 186, 30):
        if (x // 30 + y // 30) % 2: cv2.rectangle(base, (x, y), (x+20, y+20), 220, -1)
cv2.rectangle(base, (12, 12), (207, 167), 130, 2)
center = (base.shape[1] / 2, base.shape[0] / 2)

In [ ]:
def detect(gray, quality):
    pts = cv2.goodFeaturesToTrack(gray, maxCorners=200, qualityLevel=quality, minDistance=7, blockSize=5, useHarrisDetector=True, k=0.04)
    return np.empty((0, 2), np.float32) if pts is None else pts.reshape(-1, 2)

def show_corners(angle=20, quality=0.02):
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(base, M, (base.shape[1], base.shape[0]))
    points_a, points_b = detect(base, quality), detect(rotated, quality)
    views = [cv2.cvtColor(base, cv2.COLOR_GRAY2RGB), cv2.cvtColor(rotated, cv2.COLOR_GRAY2RGB)]
    for view, points in zip(views, [points_a, points_b]):
        for x, y in points: cv2.circle(view, (round(float(x)), round(float(y))), 3, (255, 60, 30), -1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, view, title in zip(axes, views, [f'原图：{len(points_a)} 点', f'旋转 {angle}°：{len(points_b)} 点']):
        ax.imshow(view); ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()

interact(
    show_corners,
    angle=IntSlider(min=0, max=60, step=5, value=20),
    quality=FloatSlider(min=0.005, max=0.15, step=0.005, value=0.02, readout_format='.3f'),
);

## 接上自己的作业

下面的实验有一部分要调用你在 `5-1_Harris响应.py` 里写的 `harris_response`。先运行这个单元把作业文件加载进来。作业还没写完也可以往下运行，用到它的单元会打印一行提示，不会报错；写完后回来重新运行这个单元。

每个实验都是先写下预测，再运行，然后把结果和原因记到 `实验记录.md`。

In [ ]:
import importlib.util, os
from pathlib import Path

CHAPTER_NAME = "05_特征点"
ASSIGNMENT = "5-1_Harris响应.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

def load_assignment():
    """加载作业文件；设置了环境变量 COURSE_IMPL_DIR 时改用那里的实现（维护者验证用）。"""
    chapter = find_chapter(CHAPTER_NAME)
    override = os.environ.get("COURSE_IMPL_DIR")
    base = chapter
    if override:
        # 相对路径先按当前目录找，找不到再按课程根目录找
        roots = [Path(override), chapter.parent / override]
        base = next((r / chapter.name for r in roots if (r / chapter.name / ASSIGNMENT).is_file()), roots[0] / chapter.name)
    spec = importlib.util.spec_from_file_location(Path(ASSIGNMENT).stem.replace("-", "_"), base / ASSIGNMENT)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，不中断 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

mod = load_assignment()   # 改了作业文件之后，重新运行这个单元再往下做
print("作业文件：", find_chapter(CHAPTER_NAME) / ASSIGNMENT)


## 实验 1：自己的响应图和 `cv2.cornerHarris` 差多少（用到作业 5-1）

先预测：

1. 两张响应图“长得”一样吗（相关系数接近 1 吗）？
2. 数值上是相等，还是差一个固定的倍数？
3. 同一张图分别用 uint8 和 float32 传给 `cornerHarris`，最大响应差多少倍？

运行后对照教程 5.2 节“自己写的响应和 OpenCV 的差多少”。倍数取决于梯度和窗口怎么缩放：Sobel 不缩放、方框窗取平均时是 256；你用了高斯窗或者别的缩放，倍数会不同，只要相关系数接近 1 就不算错。

In [ ]:
mine = attempt(mod.harris_response, base, 0.04, 3)
if mine is not None:
    reference = cv2.cornerHarris(np.float32(base), 3, 3, 0.04)
    strong = np.abs(reference) > 1e-3 * np.abs(reference).max()
    print('相关系数：', round(float(np.corrcoef(mine.ravel(), reference.ravel())[0, 1]), 4))
    print('你的响应 / cornerHarris 的响应，中位数：', round(float(np.median(mine[strong] / reference[strong])), 1))
    from_uint8 = cv2.cornerHarris(base, 3, 3, 0.04)
    print(f'cornerHarris 用 float32 输入和 uint8 输入，最大响应之比：{reference.max() / from_uint8.max():.3g}（255 的四次方是 {255.0 ** 4:.3g}）')
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    axes[0].imshow(base, cmap='gray'); axes[0].set_title('输入')
    for ax, response, title in [(axes[1], mine, '你的 harris_response'), (axes[2], reference, 'cv2.cornerHarris（float32 输入）')]:
        limit = 0.3 * float(np.abs(response).max())
        ax.imshow(response, cmap='coolwarm', vmin=-limit, vmax=limit); ax.set_title(title + '\n红色为正（角点），蓝色为负（边）')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()

## 实验 2：图像调暗之后，固定阈值还能用吗（用到作业 5-1）

把 `base` 整体乘一个增益再算响应。检出的角点数用两种阈值各数一次：固定阈值（原图最大响应的 1%）和相对阈值（每张图自己最大响应的 1%）。

先预测：增益 0.5 时最大响应变成原来的多少倍？固定阈值下，增益降到多少时角点开始变少，降到多少时一个都不剩？相对阈值检出的个数会不会变？

运行后对照教程 5.2 节“亮度”一段。下面的 `count_corners` 做了阈值加非极大值抑制，这一步不在作业里，可以直接看。

In [ ]:
def count_corners(response, threshold, radius=4):
    """阈值加非极大值抑制：比阈值大、并且是 (2*radius+1) 邻域里最大值的像素算一个角点。"""
    local_max = response == cv2.dilate(response, np.ones((2 * radius + 1, 2 * radius + 1), np.uint8))
    return cv2.connectedComponents((local_max & (response > threshold)).astype(np.uint8))[0] - 1

full = attempt(mod.harris_response, base, 0.04, 3)
if full is not None:
    fixed = 0.01 * float(full.max())
    print('增益   最大响应之比   固定阈值检出   相对阈值检出')
    for gain in (1.0, 0.7, 0.5, 0.3):
        response = mod.harris_response(base.astype(np.float32) * gain, 0.04, 3)
        print(f'{gain:<6} {float(response.max()) / float(full.max()):<14.4f} {count_corners(response, fixed):<14} {count_corners(response, 0.01 * float(response.max()))}')

## 实验 3：k 和窗口大小各管什么（用到作业 5-1）

两张小图：一条模糊过的 45 度直边，和一个直角（顶点在 (39.5, 39.5)）。

先预测：

1. 45 度直边的中段，响应是正还是负？k 从 0.04 提到 0.2，直边和直角顶点的响应各往哪个方向变？
2. 窗口从 3 加到 15，直角的响应峰值离真实顶点是更近还是更远？峰值落在直角的里面还是外面？

运行后对照教程 5.2 节“k 到底在管什么”和“窗口大小：方框窗越大，峰值离顶点越远”。这里只有一个摆正的直角，教程的表是 16 种朝向和亚像素位置的平均，数值不会相同，对照的是变化的方向。

In [ ]:
yy, xx = np.mgrid[0:96, 0:96]
edge_image = cv2.GaussianBlur(np.where(xx + yy > 95, 200, 0).astype(np.float32), (0, 0), 1.0)   # 45 度直边
corner_image = np.zeros((96, 96), np.float32)
corner_image[40:, 40:] = 200                                                                    # 直角，顶点在 (39.5, 39.5)

first = attempt(mod.harris_response, corner_image, 0.04, 5)
if first is not None:
    scale = float(first[20:60, 20:60].max())
    print('以 k=0.04、窗口 5 时直角顶点的最大响应为 1：')
    for k in (0.04, 0.1, 0.2):
        on_edge = mod.harris_response(edge_image, k, 5)[36:60, 36:60]
        on_corner = mod.harris_response(corner_image, k, 5)[20:60, 20:60]
        print(f'  k={k:<5} 直边中段的最大响应 {float(on_edge.max()) / scale:+.3f}，最小响应 {float(on_edge.min()) / scale:+.3f}；直角顶点的最大响应 {float(on_corner.max()) / scale:+.3f}')
    print('响应峰值到真实顶点 (39.5, 39.5) 的距离：')
    for window in (3, 7, 11, 15):
        response = mod.harris_response(corner_image, 0.04, window)[20:60, 20:60]
        py, px = np.unravel_index(int(np.argmax(response)), response.shape)
        print(f'  窗口 {window:<3} 峰值在 (x={px + 20}, y={py + 20})，距离 {np.hypot(px + 20 - 39.5, py + 20 - 39.5):.2f} px')

## 破坏实验与记录

- 在最上面的互动单元里提高 `qualityLevel`，记录检测点数量何时骤减。
- 比较 0°、30°、45°、60°，不要只比较点数，还要观察位置是否对应。
- 对 `base` 先做强高斯模糊，观察定位和低对比度点。
- 实验 3 里把直角换成 135 度的钝角（自己画一个），k 取 0.04 和 0.1 各看一次顶点还有没有正响应，对照教程 5.2 节的张角表。

在 `实验记录.md` 中区分“检测数量”和“可重复性”。